# 98110 Mobility Demand Analysis by Trip Purpose

**Wkg Questions:**
1. How do trip purposes (HBW, HBO, NHB) vary by time of day and corridor?
2. What % of trips outside transit hours are work-related vs other purposes?
3. Which corridors show the highest demand for each trip purpose during off-hours?
4. How might adjustments to transit services better serve specific trip purposes?

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set up formatting
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid")
comma_fmt = FuncFormatter(lambda x, p: format(int(x), ','))

## Configuration 

In [ ]:
# File paths
INPUT_FILE = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\2013907_BI_Corridor_Volume_2025\2014261_BI_Corridor_Volume_2025_dayparts_purpose.csv'
OUTPUT_DIR = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\output_trip_purpose'

# Create output subdirectories
os.makedirs(OUTPUT_DIR, exist_ok=True)
for subdir in ['heatmaps', 'summary_tables', 'analysis_reports', 'purpose_breakdowns']:
    os.makedirs(os.path.join(OUTPUT_DIR, subdir), exist_ok=True)

## Transit Operating Hours

In [ ]:
TRANSIT_CONFIG = {
    'weekday': {
        'start_hour': 4.5,
        'end_hour': 19.25,
        'days': ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday'],
        'name': 'Weekday'
    },
    'saturday': {
        'start_hour': 9,
        'end_hour': 18,
        'days': ['Saturday'],
        'name': 'Saturday'
    },
    'sunday': {
        'start_hour': 9,
        'end_hour': 16,
        'days': ['Sunday'],
        'name': 'Sunday'
    }
}

TIME_PERIODS = {
    '0: All Day (12am-12am)': 'All Day',
    '1: Early AM (12am-6am)': 'Early AM',
    '2: Peak AM (6am-10am)': 'Peak AM',
    '3: Mid-Day (10am-3pm)': 'Mid-Day',
    '4: Peak PM (3pm-7pm)': 'Peak PM',
    '5: Late PM (7pm-12am)': 'Late PM'
}

TRIP_PURPOSES = {
    'Home to Work': 'Home to Work',
    'Home to Other': 'Home to Other',
    'Non-Home Based Trip': 'Non-Home Based'
}

COMMERCIAL_CORRIDORS = [
    'HIGH_SCHOOL_RD',
    'SPORTSMAN_CLUB',
    'SR305_MID',
    'SR305_N',
    'SR305_S',
    'WINSLOW_WAY'
]

ORIGIN_ZONE = 'kitsap_broadly2'
DESTINATION_ZONE = 'kitsap_broadly4'
VOLUME_COL = 'Average Daily O-M-D Traffic (StL Volume)'

## Helper Functions

In [ ]:
def extract_hour(period_str):
    """Extract hour number from period string like '08: 7am-8am'"""
    try:
        return int(str(period_str).split(':')[0])
    except:
        return None

def get_day_category(day_type):
    """Categorize day type into weekday, Saturday, or Sunday"""
    day_str = str(day_type).lower()
    if 'monday' in day_str or 'tuesday' in day_str or 'wednesday' in day_str or \
       'thursday' in day_str or 'friday' in day_str:
        return 'weekday'
    elif 'saturday' in day_str:
        return 'saturday'
    elif 'sunday' in day_str:
        return 'sunday'
    else:
        return 'other'

def get_transit_status(hour, day_category):
    """Determine if a given hour is during transit service"""
    if pd.isna(hour) or day_category not in TRANSIT_CONFIG:
        return 'Unknown'
    config = TRANSIT_CONFIG[day_category]
    if hour >= config['start_hour'] and hour < config['end_hour']:
        return 'During Service'
    elif hour < config['start_hour']:
        return 'Before Service'
    else:
        return 'After Service'

def calculate_trip_volumes(df):
    """Calculate trip volumes from percentages"""
    df = df.copy()
    for purpose in TRIP_PURPOSES.keys():
        if purpose in df.columns:
            df[purpose] = (df[VOLUME_COL] * df[purpose]).round(0)
    return df

## Read and Prepare Data

In [ ]:
print('=' * 80)
print('98110 MOBILITY DEMAND ANALYSIS BY TRIP PURPOSE')
print('=' * 80)
print(f'\nAnalysis started: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

try:
    print(f'\nReading data from: {INPUT_FILE}')
    df = pd.read_csv(INPUT_FILE)
    print(f'  Successfully loaded {len(df):,} records')
except Exception as e:
    print(f'ERROR: Could not read input file: {e}')
    print('Please check the file path and try again.')
    raise

df = calculate_trip_volumes(df)
df['Hour'] = df['Day Part'].apply(extract_hour)
df['Day_Category'] = df['Day Type'].apply(get_day_category)
df['Transit_Status'] = df.apply(
    lambda row: get_transit_status(row['Hour'], row['Day_Category']), axis=1
)
df['Time_Period'] = df['Day Part']

df_hourly = df[df['Day Part'] != '0: All Day (12am-12am)'].copy()
df_daily = df[df['Day Part'] == '0: All Day (12am-12am)'].copy()
df_hourly_individual = df_hourly[~df_hourly['Day Type'].astype(str).str.contains('All Days', na=False)].copy()
df_daily_individual = df_daily[~df_daily['Day Type'].astype(str).str.contains('All Days', na=False)].copy()

print(f'\nData Overview:')
print(f'  Hourly records: {len(df_hourly_individual):,}')
print(f'  Daily total records: {len(df_daily_individual):,}')
print(f'  Day types: {sorted(df_hourly_individual["Day Type"].unique())}')
print(f'  Transit status distribution:')
print(df_hourly_individual['Transit_Status'].value_counts().to_string())

## Summary Tables by Trip Purpose

In [ ]:
def create_purpose_summary(data_df, period_code, period_name, purpose):
    period_data = data_df[data_df['Day Part'] == period_code]
    if len(period_data) == 0:
        return None
    summary = period_data.pivot_table(
        values=purpose,
        index='Middle Filter Zone Name',
        columns='Day Type',
        aggfunc='sum',
        fill_value=0
    )
    summary.loc['TOTAL'] = summary.sum()
    summary['TOTAL'] = summary.sum(axis=1)
    return summary

purpose_summaries = {}
print('\n' + '=' * 80)
print('CREATING TRIP PURPOSE SUMMARY TABLES')
print('=' * 80)

for purpose_name, purpose_label in TRIP_PURPOSES.items():
    print(f'\n{purpose_label}:')
    purpose_summaries[purpose_name] = {}
    for period_code, period_name in TIME_PERIODS.items():
        summary = create_purpose_summary(df_hourly_individual, period_code, period_name, purpose_name)
        if summary is not None:
            purpose_summaries[purpose_name][period_code] = summary
            filename = f'{purpose_name}_{period_name}_summary.csv'.replace(' ', '_')
            filepath = os.path.join(OUTPUT_DIR, 'summary_tables', filename)
            summary.to_csv(filepath)
            print(f'  {period_name}: {len(summary)} rows')

## Heatmaps: Hourly Patterns by Purpose by Corridor

In [ ]:
print('\n' + '=' * 80)
print('CREATING TRIP PURPOSE HEATMAPS')
print('=' * 80)

for corridor in COMMERCIAL_CORRIDORS:
    print(f'\n{corridor}:')
    corridor_data = df_hourly_individual[df_hourly_individual['Middle Filter Zone Name'] == corridor]
    if len(corridor_data) == 0:
        print('  No data found')
        continue
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for idx, (purpose_name, purpose_label) in enumerate(TRIP_PURPOSES.items()):
        ax = axes[idx]
        pivot_data = corridor_data.pivot_table(
            values=purpose_name, index='Hour', columns='Day Type', aggfunc='sum', fill_value=0
        )
        if len(pivot_data) == 0:
            ax.text(0.5, 0.5, 'No Data', ha='center', va='center')
            continue
        max_val = pivot_data.max().max()
        sns.heatmap(pivot_data, annot=True, fmt='.0f', cmap='YlOrRd',
                    linewidths=0.5, vmin=0, vmax=max_val, ax=ax,
                    cbar_kws={'label': 'Trip Volume'})
        ax.set_title(f'{purpose_label}\n(Max: {max_val:,.0f})')
        ax.set_xlabel('Day Type')
        ax.set_ylabel('Hour of Day')
    plt.suptitle(f'Trip Purpose Analysis - {corridor}', fontsize=14, y=1.02)
    plt.tight_layout()
    filename = f'Heatmap_{corridor}_purposes.pdf'
    filepath = os.path.join(OUTPUT_DIR, 'heatmaps', filename)
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()
    print('  Saved heatmap')

## Purpose Proportion Analysis

In [ ]:
print('\n' + '=' * 80)
print('TRIP PURPOSE PROPORTION ANALYSIS')
print('=' * 80)

purpose_proportions = []
for period_code, period_name in TIME_PERIODS.items():
    period_data = df_hourly_individual[df_hourly_individual['Day Part'] == period_code]
    for transit_status in ['During Service', 'Before Service', 'After Service']:
        status_data = period_data[period_data['Transit_Status'] == transit_status]
        if len(status_data) == 0:
            continue
        total_trips = status_data[list(TRIP_PURPOSES.keys())].sum().sum()
        if total_trips > 0:
            for purpose_name, purpose_label in TRIP_PURPOSES.items():
                purpose_total = status_data[purpose_name].sum()
                proportion = (purpose_total / total_trips * 100) if total_trips > 0 else 0
                purpose_proportions.append({
                    'Time_Period': period_name,
                    'Transit_Status': transit_status,
                    'Trip_Purpose': purpose_label,
                    'Volume': purpose_total,
                    'Proportion': proportion
                })

purpose_prop_df = pd.DataFrame(purpose_proportions)
print('\nTrip Purpose Proportions by Time Period and Transit Status:')
if len(purpose_prop_df) > 0:
    print(purpose_prop_df.to_string(index=False))
else:
    print('No data available for proportion analysis')

## Corridor-Specific Purpose Analysis

In [ ]:
print('\n' + '=' * 80)
print('CORRIDOR-SPECIFIC PURPOSE ANALYSIS')
print('=' * 80)

corridor_purpose_analysis = []
for corridor in COMMERCIAL_CORRIDORS:
    corridor_data = df_hourly_individual[df_hourly_individual['Middle Filter Zone Name'] == corridor]
    if len(corridor_data) == 0:
        continue
    for transit_status in ['During Service', 'Before Service', 'After Service']:
        status_data = corridor_data[corridor_data['Transit_Status'] == transit_status]
        if len(status_data) == 0:
            continue
        row = {'Corridor': corridor, 'Transit_Status': transit_status}
        total = 0
        for purpose_name, purpose_label in TRIP_PURPOSES.items():
            volume = status_data[purpose_name].sum()
            row[purpose_label] = volume
            total += volume
        if total > 0:
            for purpose_name, purpose_label in TRIP_PURPOSES.items():
                row[f'{purpose_label}_Pct'] = (row[purpose_label] / total * 100)
            corridor_purpose_analysis.append(row)

corridor_purpose_df = pd.DataFrame(corridor_purpose_analysis)
print('\nCorridor Analysis by Transit Status:')
print(corridor_purpose_df.to_string())

## Peak Hour Analysis by Purpose

In [ ]:
print('\n' + '=' * 80)
print('PEAK HOUR ANALYSIS BY PURPOSE')
print('=' * 80)

peak_hours_by_purpose = []
for purpose_name, purpose_label in TRIP_PURPOSES.items():
    for corridor in COMMERCIAL_CORRIDORS:
        corridor_data = df_hourly_individual[df_hourly_individual['Middle Filter Zone Name'] == corridor]
        if len(corridor_data) == 0:
            continue
        peak_idx = corridor_data[purpose_name].idxmax()
        peak_row = corridor_data.loc[peak_idx]
        peak_hours_by_purpose.append({
            'Trip_Purpose': purpose_label,
            'Corridor': corridor,
            'Peak_Hour': f"{int(peak_row['Hour']):02d}:00-{int(peak_row['Hour'])+1:02d}:00",
            'Peak_Volume': peak_row[purpose_name],
            'Day_Type': peak_row['Day Type'],
            'Transit_Status': peak_row['Transit_Status']
        })

peak_purpose_df = pd.DataFrame(peak_hours_by_purpose)
print('\nPeak Hours by Purpose:')
print(peak_purpose_df.to_string(index=False))

## Export Results

In [ ]:
print('\n' + '=' * 80)
print('EXPORTING RESULTS')
print('=' * 80)

excel_path = os.path.join(OUTPUT_DIR, 'complete_trip_purpose_analysis.xlsx')
with pd.ExcelWriter(excel_path) as writer:
    purpose_prop_df.to_excel(writer, sheet_name='Purpose_Proportions', index=False)
    corridor_purpose_df.to_excel(writer, sheet_name='Corridor_Purpose_Analysis', index=False)
    peak_purpose_df.to_excel(writer, sheet_name='Peak_Hours_by_Purpose', index=False)
    for purpose_name, purpose_label in TRIP_PURPOSES.items():
        hourly_pivot = df_hourly_individual.pivot_table(
            values=purpose_name,
            index=['Middle Filter Zone Name', 'Hour'],
            columns='Day Type',
            aggfunc='sum',
            fill_value=0
        )
        hourly_pivot.to_excel(writer, sheet_name=f'{purpose_name}_Hourly')

print(f'\nAll results saved to: {excel_path}')
print(f'Heatmaps saved to: {os.path.join(OUTPUT_DIR, "heatmaps")}')

## Key Findings

In [ ]:
print('\n' + '=' * 80)
print('KEY FINDINGS')
print('=' * 80)

total_by_purpose = {}
for purpose_name, purpose_label in TRIP_PURPOSES.items():
    total_by_purpose[purpose_label] = df_hourly_individual[purpose_name].sum()
total_trips = sum(total_by_purpose.values())

outside_data = df_hourly_individual[
    df_hourly_individual['Transit_Status'].isin(['Before Service', 'After Service'])
]
outside_by_purpose = {}
for purpose_name, purpose_label in TRIP_PURPOSES.items():
    outside_by_purpose[purpose_label] = outside_data[purpose_name].sum()

print('\n1. OVERALL TRIP VOLUME BY PURPOSE:')
for purpose, volume in total_by_purpose.items():
    pct = (volume / total_trips * 100) if total_trips > 0 else 0
    print(f'   - {purpose}: {volume:,.0f} ({pct:.1f}%)')

print('\n2. TRIPS OUTSIDE TRANSIT HOURS BY PURPOSE:')
outside_total = sum(outside_by_purpose.values())
for purpose, volume in outside_by_purpose.items():
    pct = (volume / outside_total * 100) if outside_total > 0 else 0
    print(f'   - {purpose}: {volume:,.0f} ({pct:.1f}%)')

print('\n3. PURPOSE COMPOSITION DURING VS OUTSIDE TRANSIT:')
for purpose, volume in total_by_purpose.items():
    outside_vol = outside_by_purpose.get(purpose, 0)
    outside_pct = (outside_vol / volume * 100) if volume > 0 else 0
    during_vol = volume - outside_vol
    during_pct = 100 - outside_pct
    print(f'   - {purpose}:')
    print(f'     During transit: {during_vol:,.0f} ({during_pct:.1f}%)')
    print(f'     Outside transit: {outside_vol:,.0f} ({outside_pct:.1f}%)')

print('\n4. CORRIDORS WITH HIGHEST OUTSIDE-TRANSIT DEMAND BY PURPOSE:')
for purpose_name, purpose_label in TRIP_PURPOSES.items():
    corridor_totals = []
    for corridor in COMMERCIAL_CORRIDORS:
        corridor_data = df_hourly_individual[df_hourly_individual['Middle Filter Zone Name'] == corridor]
        outside_corridor = corridor_data[
            corridor_data['Transit_Status'].isin(['Before Service', 'After Service'])
        ][purpose_name].sum()
        corridor_totals.append((corridor, outside_corridor))
    corridor_totals.sort(key=lambda x: x[1], reverse=True)
    print(f'\n   {purpose_label}:')
    for corridor, volume in corridor_totals[:3]:
        if volume > 0:
            print(f'     - {corridor}: {volume:,.0f} trips outside transit')

print('\n5. FOOD FOR THOUGHT:')
hbw_outside_pct = (outside_by_purpose.get('Home-Based Work', 0) / total_by_purpose.get('Home-Based Work', 1) * 100)
hbo_outside_pct = (outside_by_purpose.get('Home-Based Other', 0) / total_by_purpose.get('Home-Based Other', 1) * 100)
nhb_outside_pct = (outside_by_purpose.get('Non-Home Based', 0) / total_by_purpose.get('Non-Home Based', 1) * 100)
if hbw_outside_pct > 20:
    print(f'   - WORK TRIPS: {hbw_outside_pct:.1f}% of work trips occur outside transit hours. Opportunity for commuters?')
if hbo_outside_pct > 25:
    print(f'   - OTHER HOME-BASED TRIPS: {hbo_outside_pct:.1f}% occur outside transit hours. Shopping/errand travel.')
if nhb_outside_pct > 30:
    print(f'   - NON-HOME BASED TRIPS: {nhb_outside_pct:.1f}% occur outside transit hours. Multi-purpose trips.')

print('\n6. TO DO:')
print('   - Review purpose-specific heatmaps in the heatmaps folder')
print('   - Check complete_trip_purpose_analysis.xlsx for detailed breakdowns')
print('   - Compare purpose patterns across different corridors')
print('   - Consider targeted surveys for purposes with high outside-transit demand')

print('\n' + '=' * 80)
print(f'Finished at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('=' * 80)